In [ ]:
# analysis.ipynb (Jupyter Notebook Cells)

## 📊 1. Setup and Imports

import pandas as pd
import numpy as np
from yahooquery import Ticker
import matplotlib.pyplot as plt

# Import the core logic from the files we developed
from langgraph_agent import build_langgraph_agent
from app import BaseAgent, SMAAgent, ConservativeAgent, fetch_yahoo_history, compute_metrics

# Re-define the agent with the notebook environment's imports
class NotebookSMAAgent(SMAAgent):
    # This class ensures the agent uses the correct, updated LangGraph logic
    pass 

# Initialize the LangGraph Agent (runs the graph builder)
lg_agent = build_langgraph_agent()


## ⚙️ 2. Configuration and Data Fetching

SYMBOL = "MSFT"  # Example ticker
PERIOD = "1y"    # Backtesting duration
INITIAL_CASH = 100000.0

# Fetch Data
df = fetch_yahoo_history(SYMBOL, period=PERIOD)

if df.empty:
    print(f"Error: No data found for {SYMBOL}")
else:
    # Prepare data for agents
    df["PrevClose"] = df["Close"].shift(1)
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    
    print(f"Successfully loaded {len(df)} days of data for {SYMBOL}.")


## 🚀 3. Run Backtest

agents = [
    NotebookSMAAgent("LangGraphSMAAgent", INITIAL_CASH),
    ConservativeAgent("RuleBasedAgent", INITIAL_CASH)
]

# Backtesting Loop
for _, row in df.iterrows():
    for agent in agents:
        agent.observe(row)
        agent.reason()
        agent.act()

print("Backtest Complete.")


## 📈 4. Performance Analysis and Visualization

# Consolidate results
results = {a.name: a.portfolio_history for a in agents}
portfolio_df = pd.DataFrame(results, index=df.index)

# Calculate Scaled Price (Benchmark)
scaled_price = df["Close"] * (INITIAL_CASH / df["Close"].iloc[0])
portfolio_df["ScaledPrice"] = scaled_price

# Calculate Metrics
metrics_data = {}
for agent in agents:
    metrics_data[agent.name] = compute_metrics(portfolio_df[agent.name])

# Display Metrics Table
metrics_df = pd.DataFrame(metrics_data).T
print("\n--- Performance Metrics ---")
print(metrics_df.to_markdown())

# Plotting the Portfolio Performance
plt.figure(figsize=(14, 7))
plt.plot(portfolio_df.index, portfolio_df["LangGraphSMAAgent"], label="LangGraph SMA Agent", color='blue')
plt.plot(portfolio_df.index, portfolio_df["RuleBasedAgent"], label="Rule-Based Agent", color='orange')
plt.plot(portfolio_df.index, portfolio_df["ScaledPrice"], label="Scaled Price (Benchmark)", linestyle='--', color='gray')
plt.title(f"{SYMBOL} Portfolio Performance Over {PERIOD}")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (USD)")
plt.legend()
plt.grid(True)
plt.show()

## 🔍 5. Inspect Last 5 Actions (Reproducible Trace)

print("\n--- Last 5 Actions Trace (LangGraph Agent) ---")
trace_df = pd.DataFrame({
    "Date": df.index[-5:],
    "Action": agents[0].action_history[-5:],
    "Explanation": agents[0].explanation_history[-5:]
}).set_index("Date")
print(trace_df.to_markdown())